# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library in Python.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD definition).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, do not subscript it.

print(f"{metadata.name}: {metadata.description}")
print(f"DOI: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview

Explore available record sets, fields, and columns in the dataset schema using their `@id`. Print out these elements for reference.

In [ ]:
# List all record sets and fields with their @id
print("Available record sets in this dataset:")
for rs in metadata.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print("  Fields and columns (@id):")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field: {field.name}, @id: {field.id}, Data Type: {getattr(field, 'data_type', 'N/A')}")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"    - Column: {col.name}, @id: {col.id}, Data Type: {getattr(col, 'data_type', 'N/A')}")
    print("")
if not metadata.record_sets:
    print("No record sets found. Trying to find them through 'distribution' or print full metadata if necessary.")
    if hasattr(metadata, 'distribution'):
        print("Distributions found:")
        for dist in getattr(metadata, 'distribution', []):
            print(json.dumps(dist, indent=2))
    print("If no record sets are defined explicitly, the mlcroissant library may infer a synthetic record set for the distribution file.")

## 3. Data Extraction

If no explicit record sets are present, `mlcroissant` uses the distribution(s) defined by the dataset. We will use the default or synthesized record set id, typically in the form of `cr:RecordSet` or inferred from the package.

Load the data records from the main (or inferred) record set into a Pandas DataFrame for analysis. Since the exact `@id` of the record set is not known from metadata, we will infer or set it to `'cr:RecordSet'`.

In [ ]:
# Try to load all available record sets. If none found, fall back to cr:RecordSet
if metadata.record_sets:
    record_sets = [rs.id for rs in metadata.record_sets]
else:
    record_sets = ['cr:RecordSet']  # Default when record sets are not explicitly in schema

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field (e.g., 'MW [kDa]' for molecular weight if present) and perform some basic data processing:
- Filter records above a threshold
- Normalize the numeric field (z-score)
- Optionally group by a descriptive field (e.g., 'Description' or 'Gene Name')

Refer to field/column `@id` names in all operations.

In [ ]:
import numpy as np

# Identify the loaded record set id (use the first or default to 'cr:RecordSet')
record_set_id = record_sets[0]

df = dataframes[record_set_id]
print(f"Available columns in {record_set_id}:")
print(df.columns.tolist())

# Select a numeric field (example: 'MW [kDa]' or its @id, else choose an available numeric column)
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
if not candidate_numeric_fields:
    # Try to convert possible numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]

print("Candidate numeric fields:", candidate_numeric_fields)

# Choose the first numeric field for analysis
numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else None

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize
    normalized_field = f"{numeric_field_id}_normalized"
    filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, normalized_field]].head())

    # Try grouping by a group_field (example: 'Gene Name' or 'Description', use available text column not same as numeric)
    candidate_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization

Visualize data distributions and relationships. We'll use matplotlib to plot a histogram of the selected numeric field, and a bar plot for the grouped means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped data is available
    if group_field_id:
        top_n = 10
        plt.figure(figsize=(12,4))
        sns.barplot(data=grouped_df.sort_values(numeric_field_id, ascending=False).head(top_n), x=group_field_id, y=numeric_field_id)
        plt.title(f"Top {top_n} {group_field_id} by Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found, skipping visualization.")

## 6. Conclusion

- We successfully loaded and explored the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library and Pandas.
- We identified numeric fields and performed basic data filtering and normalization, referencing all fields by their column `@id`.
- Visualizations provide insight into distributions and key groupings in the dataset.

**You can further tailor this notebook to more specialized analysis or machine learning workflows by referring to column and field `@id` values as discovered in Step 2.**